# 08. Tabla maestra consolidada y comparabilidad del SVM

Este notebook corrige dos observaciones: la tabla maestra no se entregaba (`results/experiments_master.csv` estaba en el `.gitignore` y no estaba en el repositorio) y las filas de SVM, corridas en Colab con `cuml.svm.SVC`, no eran comparables con las demás, que usan `sklearn.svm.SVC`.

## 1. Tabla maestra de las 112 combinaciones

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import numpy as np
import pandas as pd

from src import config
from src import master_table as mt
from src import stats_comparison as sc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

master, best = mt.save_master_tables()
print(f"Tabla maestra: {len(master)} filas -> {mt.MASTER_112_PATH.relative_to(config.PROJECT_ROOT)}")
print(f"Selección por modelo: {len(best)} filas -> {mt.BEST_PER_MODEL_PATH.relative_to(config.PROJECT_ROOT)}")
print(f"Diferencia crítica de Nemenyi (k = {len(master)}, 3 folds): {master.attrs['critical_difference']:.1f}")
master.groupby(["model", "source"]).size().unstack(fill_value=0)

Tabla maestra: 112 filas -> results\master_table_112.csv
Selección por modelo: 7 filas -> results\best_per_model.csv
Diferencia crítica de Nemenyi (k = 112, 3 folds): 115.4


C:\Users\valef\anaconda3\envs\creditcard\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


source,colab_gpu_cuml,local_cpu_sklearn
model,,
decision_tree,0,16
knn,0,16
logistic_regression,0,16
naive_bayes,0,16
random_forest,0,16
svm,16,0
xgboost,0,16


La tabla `results/master_table_112.csv` une las 96 filas locales (`experiments_master.csv`), las 16 de SVM (`svm_gpu_results_colab.csv`) y las métricas complementarias (`experiments_posthoc_metrics.csv`). Cada fila trae el origen del cálculo (`source`), la semilla, el tiempo y las métricas por fold con su media y desviación estándar entre los 3 folds externos, **con `ddof=1` en todas** (la tabla original usaba `ddof=0`, que con 3 folds subestima la dispersión). También trae el rango medio de F1 entre las 112 combinaciones y si la combinación queda a menos de la diferencia crítica de Nemenyi (115.4) de la mejor. Los tiempos de SVM son de GPU y se marcan con `time_comparable = False`. El PR-AUC de SVM viene del reajuste en CPU (ver la columna `pr_auc_source`). A continuación, las 15 primeras filas ordenadas por rango medio de F1.

In [2]:
columns = ["model", "technique", "method", "source", "random_state", "total_time_seconds", "time_comparable",
           "f1_mean", "f1_std", "precision_mean", "recall_mean", "roc_auc_mean", "roc_auc_std", "pr_auc_mean",
           "f1_avg_rank", "within_nemenyi_cd_of_best"]
master[columns].head(15).round(4)

,model,technique,method,source,random_state,total_time_seconds,time_comparable,f1_mean,f1_std,precision_mean,recall_mean,roc_auc_mean,roc_auc_std,pr_auc_mean,f1_avg_rank,within_nemenyi_cd_of_best
0,random_forest,class_weight,genetic_deap,local_cpu_sklearn,42,2432.4055,True,0.5460,0.0108,0.4980,0.6042,0.7813,0.0068,0.5580,3.0000,True
1,random_forest,class_weight,random_search,local_cpu_sklearn,42,163.6313,True,0.5453,0.0108,0.4951,0.6071,0.7820,0.0055,0.5573,3.3333,True
2,xgboost,class_weight,bayesian_optuna,local_cpu_sklearn,42,94.3536,True,0.5451,0.0122,0.4896,0.6154,0.7817,0.0050,0.5545,4.0000,True
3,xgboost,class_weight,grid_search,local_cpu_sklearn,42,250.5575,True,0.5444,0.0064,0.4972,0.6042,0.7800,0.0002,0.5500,5.0000,True
4,random_forest,class_weight,grid_search,local_cpu_sklearn,42,178.2845,True,0.5448,0.0091,0.4920,0.6103,0.7833,0.0047,0.5579,5.0000,True
5,random_forest,class_weight,bayesian_optuna,local_cpu_sklearn,42,344.0003,True,0.5444,0.0097,0.4998,0.5980,0.7804,0.0068,0.5567,5.3333,True
6,xgboost,class_weight,random_search,local_cpu_sklearn,42,86.7852,True,0.5438,0.0114,0.4964,0.6017,0.7802,0.0038,0.5517,6.6667,True
7,xgboost,class_weight,genetic_deap,local_cpu_sklearn,42,335.5479,True,0.5440,0.0127,0.4946,0.6048,0.7807,0.0036,0.5576,8.0000,True
8,svm,class_weight,genetic_deap,colab_gpu_cuml,42,1944.8928,False,0.5385,0.0063,0.4753,0.6214,0.7635,0.0111,0.5065,11.6667,True
9,svm,class_weight,grid_search,colab_gpu_cuml,42,220.3367,False,0.5383,0.0058,0.4745,0.6228,0.7640,0.0112,0.5074,11.6667,True


In [3]:
# Control de integridad: una fila por combinación, sin vacíos en las columnas clave.
assert len(master) == 112 and not master.duplicated(["model", "technique", "method"]).any()
assert master[["f1_mean", "f1_std", "roc_auc_mean", "total_time_seconds"]].notna().all().all()
print("OK: 112 combinaciones únicas; F1, ROC-AUC y tiempo completos.")
print("Filas sin PR-AUC:", int(master["pr_auc_mean"].isna().sum()))

OK: 112 combinaciones únicas; F1, ROC-AUC y tiempo completos.
Filas sin PR-AUC: 0


## 2. Mejor combinación por modelo y motivo de la selección

Para cada modelo se elige la combinación (técnica de balanceo y método de optimización) con mayor F1 medio en el bucle externo. La tabla se guarda en `results/best_per_model.csv`, con el motivo de cada selección. La columna `within_nemenyi_cd_of_global_best` indica si la combinación está dentro del conjunto de finalistas de la comparación de Friedman y Nemenyi (a menos de la diferencia crítica de la mejor global).

In [4]:
best[["model", "technique", "method", "f1_mean_std", "roc_auc_mean_std", "gap_to_runner_up",
      "runner_up", "within_nemenyi_cd_of_global_best", "is_global_best", "source"]].round(4)

,model,technique,method,f1_mean_std,roc_auc_mean_std,gap_to_runner_up,runner_up,within_nemenyi_cd_of_global_best,is_global_best,source
0,random_forest,class_weight,genetic_deap,0.5460 ± 0.0108,0.7813 ± 0.0068,0.0006,class_weight|random_search,True,True,local_cpu_sklearn
1,xgboost,class_weight,bayesian_optuna,0.5451 ± 0.0122,0.7817 ± 0.0050,0.0007,class_weight|grid_search,True,False,local_cpu_sklearn
2,svm,class_weight,genetic_deap,0.5385 ± 0.0063,0.7635 ± 0.0111,0.0001,class_weight|bayesian_optuna,True,False,colab_gpu_cuml
3,knn,class_weight,bayesian_optuna,0.5266 ± 0.0062,0.7622 ± 0.0035,0.0000,class_weight|random_search,True,False,local_cpu_sklearn
4,decision_tree,class_weight,genetic_deap,0.5190 ± 0.0078,0.7528 ± 0.0055,0.0011,smote|bayesian_optuna,True,False,local_cpu_sklearn
5,logistic_regression,smote,genetic_deap,0.5167 ± 0.0135,0.7467 ± 0.0127,0.0013,smote|grid_search,True,False,local_cpu_sklearn
6,naive_bayes,smote,genetic_deap,0.4993 ± 0.0048,0.7450 ± 0.0051,0.0000,smote|grid_search,True,False,local_cpu_sklearn


In [5]:
for _, r in best.iterrows():
    print(f"- {r['model']}: {r['selection_reason']}")

- random_forest: Mayor F1 medio externo entre las 16 combinaciones de random_forest; ventaja de 0.0006 sobre class_weight|random_search (0.1 desviaciones estándar entre folds).
- xgboost: Mayor F1 medio externo entre las 16 combinaciones de xgboost; ventaja de 0.0007 sobre class_weight|grid_search (0.1 desviaciones estándar entre folds).
- svm: Mayor F1 medio externo entre las 16 combinaciones de svm; ventaja de 0.0001 sobre class_weight|bayesian_optuna (0.0 desviaciones estándar entre folds).
- knn: Mayor F1 medio externo entre las 16 combinaciones de knn; ventaja de 0.0000 sobre class_weight|random_search (0.0 desviaciones estándar entre folds).
- decision_tree: Mayor F1 medio externo entre las 16 combinaciones de decision_tree; ventaja de 0.0011 sobre smote|bayesian_optuna (0.1 desviaciones estándar entre folds).
- logistic_regression: Mayor F1 medio externo entre las 16 combinaciones de logistic_regression; ventaja de 0.0013 sobre smote|grid_search (0.1 desviaciones estándar entre 

**Lectura.**
- La mejor combinación global es Random Forest con `class_weight` y `genetic_deap` (F1 0.5460 ± 0.0108), seguida de XGBoost con `class_weight` y `bayesian_optuna` (0.5451 ± 0.0122).
- Las siete combinaciones elegidas están dentro de la diferencia crítica de la mejor global. Esto **no prueba equivalencia**: con 3 folds la prueba de Nemenyi no tiene potencia (la máxima diferencia de rangos, 107.7, es menor que la diferencia crítica, 115.4), así que no puede separar ninguna pareja.
- La ventaja de cada combinación elegida sobre la segunda mejor de su mismo modelo es de entre 0.0000 y 0.0013 de F1, mucho menor que la desviación entre folds. La elección dentro de cada modelo es prácticamente un desempate. En cinco de los siete modelos la técnica elegida es `class_weight` y en los otros dos es SMOTE.

## 3. ¿Son comparables las 16 filas de SVM?

Repetir las búsquedas de SVM en CPU no es viable (la corrida local de `svm/none/genetic_deap` tardó más de 4 horas). Lo que sí se puede verificar es la parte de la comparación que depende de la evaluación y no de la búsqueda:

1. Se toman los hiperparámetros que eligió el bucle interno de Colab en cada fold externo.
2. Se reajusta, con el mismo pipeline del resto del proyecto (`balancing.build_pipeline`, con `sklearn.svm.SVC`), sobre los mismos folds externos, y se mide el F1 de prueba.
3. Se compara con el F1 por fold que reportó Colab.

Si los dos coinciden, las filas de SVM miden lo mismo que las demás. Lo que **no** se puede verificar con esto es que la búsqueda de cuML habría elegido los mismos hiperparámetros que la de libsvm, ni los tiempos.

In [6]:
check = mt.load_svm_check()
if check is None or len(check) < 16:
    raise RuntimeError("Falta el reajuste local de SVM: correr `python -m src.svm_comparability`.")
check = check.reset_index()
cmp = check[["technique", "method", "f1_colab_mean", "f1_local_mean", "f1_max_abs_diff_per_fold",
             "roc_auc_max_abs_diff_per_fold"]].copy()
cmp["dif_media_f1 (local - colab)"] = cmp["f1_local_mean"] - cmp["f1_colab_mean"]
cmp = cmp.sort_values(["technique", "method"]).reset_index(drop=True)
display(cmp.round(4))

diff = cmp["dif_media_f1 (local - colab)"]
by_tech = cmp.groupby("technique")["dif_media_f1 (local - colab)"].mean().round(4)
print(f"Diferencia media de F1 (local - Colab), 16 filas: {diff.mean():+.4f} (mín {diff.min():+.4f}, máx {diff.max():+.4f})")
print("Por técnica:", by_tech.to_dict())
print(f"Máxima diferencia absoluta por fold: {cmp['f1_max_abs_diff_per_fold'].max():.4f}")
print(f"Filas con diferencia por fold mayor que 0.01: {(cmp['f1_max_abs_diff_per_fold'] > 0.01).sum()} de 16")
print(f"Mediana de la desviación estándar entre folds del F1 (resto de modelos): "
      f"{master.loc[master['model'] != 'svm', 'f1_std'].median():.4f}")

,technique,method,f1_colab_mean,f1_local_mean,f1_max_abs_diff_per_fold,roc_auc_max_abs_diff_per_fold,dif_media_f1 (local - colab)
0,adasyn,bayesian_optuna,0.5298,0.5376,0.0127,0.0008,0.0078
1,adasyn,genetic_deap,0.5295,0.5370,0.0136,0.0008,0.0075
2,adasyn,grid_search,0.5290,0.5365,0.0112,0.0008,0.0074
3,adasyn,random_search,0.5295,0.5377,0.0110,0.0005,0.0082
4,class_weight,bayesian_optuna,0.5384,0.5430,0.0075,0.0000,0.0046
5,class_weight,genetic_deap,0.5385,0.5422,0.0072,0.0000,0.0036
6,class_weight,grid_search,0.5383,0.5442,0.0079,0.0000,0.0059
7,class_weight,random_search,0.5372,0.5434,0.0069,0.0000,0.0062
8,none,bayesian_optuna,0.4378,0.4521,0.0160,0.0002,0.0143
9,none,genetic_deap,0.4390,0.4516,0.0161,0.0003,0.0126


Diferencia media de F1 (local - Colab), 16 filas: +0.0080 (mín +0.0036, máx +0.0171)
Por técnica: {'adasyn': 0.0077, 'class_weight': 0.0051, 'none': 0.0143, 'smote': 0.005}
Máxima diferencia absoluta por fold: 0.0308
Filas con diferencia por fold mayor que 0.01: 8 de 16
Mediana de la desviación estándar entre folds del F1 (resto de modelos): 0.0061


**Lectura.**
- **Las filas de SVM de Colab no reproducen exactamente el F1 de scikit-learn.** Con los mismos hiperparámetros, los mismos folds y el mismo pipeline, el F1 reajustado en CPU es mayor en las 16 filas, en promedio en 0.008 (entre 0.004 y 0.017). La diferencia es mayor sin balanceo (0.014) y menor con `class_weight` y SMOTE (0.005). En 8 de las 16 filas hay al menos un fold con diferencia mayor que 0.01. La mediana de la desviación entre folds de los demás modelos es 0.006, de modo que la diferencia es del orden de una desviación estándar.
- El ROC-AUC casi no cambia (diferencia máxima por fold de 0.001), así que el orden en que el modelo clasifica a los clientes es el mismo y la discrepancia viene del punto de corte de las predicciones. Es consistente con que el solver y la calibración de probabilidades de cuML no son idénticos a los de libsvm, pero es una hipótesis que no se verificó.
- **Conclusión:** las filas de SVM de Colab no son estrictamente comparables en F1 con las demás, y el sesgo siempre va en contra de SVM. Los tiempos tampoco son comparables. Por eso el análisis siguiente se hace con tres versiones de la comparación.

## 4. Sensibilidad de la comparación estadística al tratamiento de SVM

Se repite la prueba de Friedman sobre el F1 por fold en tres escenarios: **A**, las 112 filas tal como se reportaron; **B**, solo las 96 combinaciones locales, sin SVM; **C**, las 112 filas con el F1 por fold de SVM reemplazado por el del reajuste en CPU. Si las conclusiones no cambian entre escenarios, no dependen del tratamiento de SVM.

In [7]:
def friedman_summary(table, label):
    matrix = sc.build_performance_matrix_from_table(table, metric="f1")
    fr = sc.friedman_test(matrix)
    cd = sc.compute_critical_difference(k=fr["k_combinations"], n_blocks=fr["n_blocks"])
    ranks = fr["avg_ranks"]
    top = ranks.index[:3].tolist()
    spread = float(ranks.max() - ranks.min())
    return {
        "escenario": label, "combinaciones": fr["k_combinations"],
        "chi2": fr["statistic"], "p": fr["p_value"], "CD Nemenyi": cd,
        "máx. dif. de rangos": spread, "Nemenyi con potencia": bool(spread > cd),
        "mejor": top[0], "2.º": top[1], "3.º": top[2],
    }, ranks


# A: las 112 filas tal como se reportaron (SVM de Colab).
A = master
# B: solo las 96 combinaciones locales (sin SVM).
B = master[master["model"] != "svm"]
# C: las 112 filas, con el F1 por fold de SVM reemplazado por el del reajuste en CPU.
C = master.copy()
is_svm = C["model"] == "svm"
C.loc[is_svm, "f1_per_fold"] = C.loc[is_svm, "f1_cpu_refit_per_fold"]
C.loc[is_svm, "f1_mean"] = C.loc[is_svm, "f1_cpu_refit_mean"]

rows, ranks_by_scenario = [], {}
for label, table in [("A. 112 filas, SVM de Colab", A), ("B. 96 filas, sin SVM", B), ("C. 112 filas, SVM reajustado en CPU", C)]:
    summary, ranks = friedman_summary(table, label)
    rows.append(summary)
    ranks_by_scenario[label] = ranks
sens = pd.DataFrame(rows).set_index("escenario")
sens

,combinaciones,chi2,p,CD Nemenyi,máx. dif. de rangos,Nemenyi con potencia,mejor,2.º,3.º
escenario,,,,,,,,,
"A. 112 filas, SVM de Colab",112,323.167000,1.339592e-22,115.378896,107.666667,False,random_forest|class_weight|genetic_deap,random_forest|class_weight|random_search,xgboost|class_weight|bayesian_optuna
"B. 96 filas, sin SVM",96,273.994204,2.898848e-19,97.457570,91.666667,False,random_forest|class_weight|genetic_deap,random_forest|class_weight|random_search,xgboost|class_weight|bayesian_optuna
"C. 112 filas, SVM reajustado en CPU",112,323.612726,1.154812e-22,115.378896,105.000000,False,random_forest|class_weight|genetic_deap,random_forest|class_weight|random_search,svm|class_weight|grid_search


In [8]:
# Ranking de las mejores combinaciones de cada modelo en cada escenario (mejor F1 medio por modelo).
def best_by_model(table):
    idx = table.groupby("model")["f1_mean"].idxmax()
    return table.loc[idx].sort_values("f1_mean", ascending=False)[["model", "technique", "method", "f1_mean"]]

for label, table in [("A", A), ("B", B), ("C", C)]:
    print(f"\nEscenario {label}:")
    display(best_by_model(table).reset_index(drop=True).round(4))


Escenario A:


,model,technique,method,f1_mean
0,random_forest,class_weight,genetic_deap,0.5460
1,xgboost,class_weight,bayesian_optuna,0.5451
2,svm,class_weight,genetic_deap,0.5385
3,knn,class_weight,bayesian_optuna,0.5266
4,decision_tree,class_weight,genetic_deap,0.5190
5,logistic_regression,smote,genetic_deap,0.5167
6,naive_bayes,smote,genetic_deap,0.4993



Escenario B:


,model,technique,method,f1_mean
0,random_forest,class_weight,genetic_deap,0.5460
1,xgboost,class_weight,bayesian_optuna,0.5451
2,knn,class_weight,bayesian_optuna,0.5266
3,decision_tree,class_weight,genetic_deap,0.5190
4,logistic_regression,smote,genetic_deap,0.5167
5,naive_bayes,smote,genetic_deap,0.4993



Escenario C:


,model,technique,method,f1_mean
0,random_forest,class_weight,genetic_deap,0.5460
1,xgboost,class_weight,bayesian_optuna,0.5451
2,svm,class_weight,grid_search,0.5442
3,knn,class_weight,bayesian_optuna,0.5266
4,decision_tree,class_weight,genetic_deap,0.5190
5,logistic_regression,smote,genetic_deap,0.5167
6,naive_bayes,smote,genetic_deap,0.4993


**Lectura.**
- **La conclusión estadística no cambia.** En los tres escenarios Friedman es significativo (p menor que 1e-18) y Nemenyi no tiene potencia: la máxima diferencia de rangos medios (107.7, 91.7 y 105.0) es menor que la diferencia crítica (115.4, 97.5 y 115.4). Las dos primeras combinaciones son las mismas en los tres escenarios: Random Forest con `class_weight` y `genetic_deap`, y Random Forest con `class_weight` y `random_search`.
- **El ranking de modelos cambia solo en la posición de SVM.** Con el SVM de Colab (escenario A), la mejor combinación de SVM tiene F1 0.5385 y queda tercera entre los modelos, detrás de Random Forest y XGBoost. Con el reajuste en CPU (escenario C) sube a 0.5442 (`class_weight` con `grid_search`), a menos de 0.001 de XGBoost (0.5451), y la tercera combinación del ranking global pasa de XGBoost con `bayesian_optuna` a SVM con `class_weight` y `grid_search`. Este cambio es desfavorable para la versión anterior del informe, que subestimaba a SVM, y se reporta así. Random Forest sigue siendo el mejor, y la diferencia con SVM (0.0018) está muy por debajo de la desviación entre folds.
- Sin SVM (escenario B) el orden de los otros seis modelos es el mismo que con SVM.

## Limitaciones

- El reajuste en CPU usa los hiperparámetros que eligió Colab. No demuestra que una búsqueda en CPU habría elegido lo mismo, así que el F1 reajustado es el de la mejor configuración encontrada por cuML evaluada con libsvm, no el de una búsqueda completa en CPU.
- El F1 reajustado se calculó con `probability=False` (las predicciones de `SVC` usan el signo de la función de decisión en ambos casos). El ROC-AUC y el PR-AUC de SVM en este reajuste salen de `decision_function` y no de `predict_proba`.
- En la tabla maestra se conservan los valores de Colab como fueron reportados (columnas `f1_*`) y el reajuste va aparte (`f1_cpu_refit_*`), para no reescribir resultados originales.
- Con 3 folds, ni la prueba de Friedman ni la de Nemenyi pueden declarar diferencias entre combinaciones cercanas, y los intervalos de las métricas son anchos. Todo lo anterior es descriptivo.